In [9]:
import secrets
from ipaddress import ip_address
from operator import truediv
from pathlib import Path
from wsgiref import util

from requests.compat import numeric_types

'''

Scenario :
AIS monitors traffic coming in and out of the local network at an IP level and identifies malformed packages

How the AIS works :
Lymphocytes detect patterns not in the self set (= detector set = unusual packages)
When enough of the lymphocytes detectors detect elements of the detector set
Then the lymphocytes activation threshold is reached which triggers an alarm

Plan for gerüst :
1. write lymphocyte class
    1a. create antibody class
        a.set up detects method specifics of detects
            a1. Implement detects based on packages in scapy this is implemented when detects is true when antibody = package
                - decide on maths
                    - determine level of threshhold for detection = stimlation = how similar does antibody have to be to package to trigger alarm
                    - decide what datastructures are we using and how is the comparison performed
                    - in the first version use maths from demo (if it can be used on larger packages)
2. create self set based on normal packages in scapy
    a. create example packets to test that the normal_traffic and antibody connect correctly
    b. download a file (write down details of training data to account for biases later) with example packets (for malicious and not malicious) and use them as example traffic (download, filter for http/https, convert to bits)
3. write the self set (lyphocytes that dont detect normal_packets) using negative selection

Future Improvements :
-> improve formal standards (ask ai)
-> improve runtime (ex. using bitarrays instead of np array)
-> fix the bitarray numpy mess
-> make convert_collection and check_conversion and the constructor of Normal_Traffic capable of handling more than 3 IP packet feilds, stop or secure the way they rely on the order being input correctly
-> create a virtual environment to send the packets and lympocytes through using mininet so that lymphocites are trained in real time
    -> change the packages/traffic based on the latest threats?
-> add lymphocites dying like the architecture.. paper and being replaced - a population of lyphocytes
-> add memory detectors (and limit them) like the architecture .. paper
-> model primary response (lymphocytes that have detected a new pathogen multiply because the others dont have the antibodies to recognise it)  and secondary response based on this paper https://www.researchgate.net/profile/Steven-Hofmeyr/publication/12197865_Architecture_for_an_Artificial_Immune_System/links/00b7d538f8550ca681000000/Architecture-for-an-Artificial-Immune-System.pdf (generally do it like that paper to improve)
-> make it so sensitivity level mimics cytokines (signaling other lymphocytes) if there were already activations in the region?
-> Have ABNORMAL_HTTP_TRAFFIC simulate not just malformed packages but also malicious packages that follow the rules of the protocol packets but simulate : IPV4 packet fragmentation attacks, TCP Null scan, TCP FIN scan, TCP XMas Scan, TCP flag manipulation, TCP sequence number anomalies, ICMP based attacks, DNS tunneling, DNS poisoning, DHCP spoofing, protocol fuzzing
-> define the threshold of activation not for once lyphocyte but as a group so that - for example - it the AIS is deployed across several vlans of a company network their joint activation thresshold enables us to recognise attack patterns across several
vlans (that use different firwalls and siems so that the detection of overall patterns is an isusue).
-> instead of making the generated antibodies random make them more like possible malicious packages without narrowing down the detector coverage

Distant Future Improvements
-> work out your own problem representation for how antigens are constructed (see review paper https://staff.fmi.uvt.ro/~daniela.zaharie/am2016/proiecte/tehnici/AIS/AIS_advances.pdf)
-> use flow records instead of raw packages (so the relationship between the packages can be considered)
-> inspect packets at TCP and DNS level
'''

'\n\nScenario :\nAIS monitors traffic coming in and out of the local network at an IP level and identifies malformed packages\n\nHow the AIS works :\nLymphocytes detect patterns not in the self set (= detector set = unusual packages)\nWhen enough of the lymphocytes detectors detect elements of the detector set\nThen the lymphocytes activation threshold is reached which triggers an alarm\n\nPlan for gerüst :\n1. write lymphocyte class\n    1a. create antibody class\n        a.set up detects method specifics of detects\n            a1. Implement detects based on packages in scapy this is implemented when detects is true when antibody = package\n                - decide on maths\n                    - determine level of threshhold for detection = stimlation = how similar does antibody have to be to package to trigger alarm\n                    - decide what datastructures are we using and how is the comparison performed\n                    - in the first version use maths from demo (if i

In [11]:
import random
import warnings
from scapy.layers.inet import IP, TCP
from scapy.packet import Packet
from enum import Enum, auto
import ipaddress
from bitarray import bitarray as BitArray
import pandas as pd
from scapy.utils import PcapReader
import numpy as np
import pytest
from numpy.typing import NDArray

"""Defines the type of traffic : all packets in this traffic are the first packets of a TCP connection exchange where SYN = 1 and ACK = 0 : these packets are used to establish connections devices inside and outside the network"""

class PacketCollectionType(Enum):
    NORMAL = auto()
    ABNORMAL = auto()
    MIXED = auto()

packet_collections = {
    PacketCollectionType.NORMAL,
    PacketCollectionType.ABNORMAL,
    PacketCollectionType.MIXED
    }


"""this class provides processes csv files into binary files that can be used by AIS """

class Packet_Collection_Generator:
    """these packets have syn = 1 so they are the first in an exchange they were extracted from pcapd files created by monitoring real world network traffic"""
    self_set_filepath = "initial_syn.csv"
    mixed_traffic_filepath = "mixed_traffic.csv"
    malicious_packets = pd.read_csv("malicious_packet_numbers.csv", header = None)
    #print(f"malicious packets are", str(malicious_packets))
    num_packets_in_collection = 2 #rows
    num_bits_in_packet = 80 #column
    Self_Set_Packets = np.empty((num_packets_in_collection, num_bits_in_packet), dtype = np.bool_)
    Abnormal_Packet_Collection = np.empty((num_packets_in_collection, num_bits_in_packet), dtype = np.bool_)
    Mixed_Packet_Collection = np.empty((num_packets_in_collection, num_bits_in_packet), dtype = np.bool_)

    """This function creates a list representing the self set. Any detector the detect method determines too close to an element of the self set is deleted"""
    def create_packet_set(self,packet_collection : PacketCollectionType,src_IP, dst_IP,dport):
            #TODO: call a method that creates the packages and save them as a WHAT?
            #only save/use (source host IP address,destination host IP address,TCP service/port number) LISYS
            """"paper class these a datapath triple : so it describes a kind of connection """
            if packet_collection == PacketCollectionType.NORMAL:
                self_set : list[Packet] = []
                while len(self_set) < self.num_packets_in_collection:
                    packet = IP(src=random.choice(src_IP),dst = random.choice(dst_IP))/TCP(sport = random.choice(dport))
                    self_set.append(packet)
                    #check that self set is a list of valid packages
                    #convert self_set to binary
                return self_set
            elif packet_collection == PacketCollectionType.ABNORMAL:
                pass
            elif packet_collection == PacketCollectionType.MIXED:
                pass
    #TODO the list[list] is a list of lists packets each item is a feild. turn into numpy array in final version
    #TODO write an indepentend converter with different logic to have values to compare the packet to
    """sampling random packets for errors in the conversion function"
    bit conversion list is the list produced by convert_collection_to_bits()"""""
    def check_bit_conversion(self,bit_conversion_list : np.ndarray, packet_collection : list[list]):
        #the list in the list[list] represents one packet
        random_packet_index= random.randrange(0, len(packet_collection))
        random_packet = packet_collection[random_packet_index]
        #taking the packet we are comparing from the np array
        comparison_packet = bit_conversion_list[random_packet_index]
        src_ip = BitArray(ipaddress.IPv4Address(random_packet[0]).packed)
        dst_ip = BitArray(ipaddress.IPv4Address(random_packet[1]).packed)
        sport = BitArray(format(random_packet[2], "016b"))
        manual_packet_bits = np.array(src_ip.tolist() + dst_ip.tolist() + sport.tolist(),dtype = np.bool_)
        "np equal checks the objects have the same shape and the same values at the corresponding index : does not require the same dtype"
        assert np.array_equal(manual_packet_bits, comparison_packet), f"packet {random_packet_index} is different from manual calculation manual type is , {type(manual_packet_bits)} while collection packet_collection type is {type(comparison_packet)} manual dtype is {manual_packet_bits.dtype} while packet_collection dtype is {comparison_packet.dtype} manual shape is {manual_packet_bits.shape} while the shape of packet_collection_bits is {comparison_packet.shape} the values is in manual packet bits are  {manual_packet_bits} while the values in packet collection bits are {comparison_packet}"
        return np.array_equal(manual_packet_bits, comparison_packet)

    """filters out unsuable packets (IPv6 or not first in the connect) that are in the packet_collection list IN THE CORRECT ORDER (scr_ip, dst_ip and sport) and converts the usable packets to numpy bit arrays which are added to 2D numbpy bit arrays that are either Normal, Abnormal or Mixed"""
    def convert_collection_to_bits(self,packet_collection : list[list]) -> np.ndarray:
        list_number = 0
        element_number = 0
        packet_collection_in_bits = []
        packet_in_bits= []
        #step 1 turn each value into a 2D numpy array
        #length of packet_collection is the number of packets (so lists) in the list
        #TODO : turn this into a for loop
        while list_number < len(packet_collection):
            #not using np.array immediately to be more computationally efficient
           # if not is_packet_usable(packet_collections[list_number]):
              #  ++element_number
            #sport is allways the third field ensuring order stays consistent
            if type(packet_collection[list_number][element_number]) == int :
                if element_number == 2:
                    sport_bits = BitArray(format(packet_collection[list_number][element_number],"016b"))
                    packet_in_bits.extend(sport_bits)
                    if list_number == 0 and element_number == 2:
                        print(("sport ip is",sport_bits, "in bits"))
                    element_number += 1
                    #if we have reached the third element we are at the end of the packet/list
                    #so we go to a new list
                    list_number += 1
                    # and we reset element number for the next list
                    element_number = 0
                    #turn finished packet into np array
                    packet_in_bits_array = np.array(packet_in_bits)
                    #clear packet for next iteration
                    packet_in_bits = []
                    #add np array containing binary packet to the collection
                    packet_collection_in_bits.append(packet_in_bits_array)
                else:
                    warnings.warn("third element of tuple should contain an int and should contain sport currently third element contains" + str(packet_collection[list_number][element_number]))
            #if the field is an ip address
            elif type(packet_collection[list_number][element_number]) == str :
                if element_number == 0:
                    src_ip_bytes = ipaddress.IPv4Address(packet_collection[list_number][element_number]).packed
                    src_ip_bits = BitArray()
                    src_ip_bits.frombytes(src_ip_bytes)
                    packet_in_bits.extend(src_ip_bits)
                    if list_number == 0 and element_number == 0 :
                        print(("src ip is ", src_ip_bits,"in bits"))
                    element_number +=1
                if element_number == 1:
                    dst_ip_bytes = ipaddress.IPv4Address(packet_collection[list_number][element_number]).packed
                    dst_ip_bits = BitArray()
                    dst_ip_bits.frombytes(dst_ip_bytes)
                    packet_in_bits.extend(dst_ip_bits)
                    if list_number == 0 and element_number == 1 :
                        print(("dst ip is",dst_ip_bits, "in bits"))
                    element_number += 1
        packet_collection_in_bits_array = np.array(packet_collection_in_bits, dtype = np.bool_)
        self.check_bit_conversion(packet_collection_in_bits_array,packet_collection)
        return packet_collection_in_bits_array

    def get_collection_in_bits_from_file(self,filepath : str) -> NDArray[np.bool_]:
        new_collection = pd.read_csv(filepath, nrows = 10000)
        print(new_collection.shape)
        new_collection_bits = self.convert_collection_to_bits(new_collection.values.tolist())
        return new_collection_bits

    """"to compare how many and which packets a different IDS flagged"""
    def comparison_IDS(self):
        pass

    """" the fields of the IP packet that we are using to identify malicious traffic are attributes fo the traffic type"""
    #TODO : make sure packets can only be added it the correct order that is the order that they appear in in the IP packet
    def __init__(self, filepath : str):
        self.packet_collection_generated = self.get_collection_in_bits_from_file(filepath)
        #self.src_IP : list[str] = ["192.168.1.1", "192.168.1.2", "192.168.1.3", "192.168.1.4", "192.168.1.5"]
        #self.dst_IP : list[str] = ["192.168.1.10", "192.168.1.20", "192.168.1.30", "192.168.1.40", "192.168.1.50"]
        #self.sport : list[int] = [49152]
        #use this to test different values in the packet fields and using actual packets
        #self.self_set : list[Packet] = create_packet_set(PacketCollectionType.NORMAL,self.src_IP,self.dst_IP,self.sport)
        #use this to test if the packet list can be connected to the bit converter and the rest of the program
        #self.dummy_self_set : list[list] = [["192.168.1.1","192.168.1.10", 49152],["192.168.1.2","192.168.1.20", 49152]]

    def __repr__(self) -> str:
        return f"{type(self).__name__}(self set = {self.packet_collection_generated})"



Self_Set_Packets = Packet_Collection_Generator(Packet_Collection_Generator.self_set_filepath).packet_collection_generated
Mixed_Packet_Collection = Packet_Collection_Generator(Packet_Collection_Generator.mixed_traffic_filepath).packet_collection_generated
#first_collection= self_set_object.dummy_self_set
#Normal_Packet_Collection = convert_collection_to_bits(first_collection)
#print("manually converted", "src_IP", BitArray(ipaddress.IPv4Address("192.168.1.1").packed),"dst_IP",BitArray(ipaddress.IPv4Address("192.168.1.10").packed), "sport", format(49152, "016b"))
#convert_collection_to_bits(first_collection)
#"in scapy you access layers by class"
#src_value = firstpackage[IP].src
#print(type(src_value))
#print(src_value)
#repr(self_set_object)



(10000, 3)
('src ip is ', bitarray('11000000101010000001001000000100'), 'in bits')
('dst ip is', bitarray('00010100001010100100000101011011'), 'in bits')
('sport ip is', bitarray('1111011001110001'), 'in bits')
(2996, 3)
('src ip is ', bitarray('11000000101010000000101000001110'), 'in bits')
('dst ip is', bitarray('01000001001101110010110001101101'), 'in bits')
('sport ip is', bitarray('1110011011111111'), 'in bits')


In [16]:

import bitarray
import pytest
import numpy as np
import random

"AIS classes : antibody and lymphocyte are defined below"


class Antibody:
    """activation threshold
-> small threshold  - > few false postives, false negatives likely
-> large threshold - > false positives likely, few false negatives
number of contiguous bits
properties of r : we are looking for the optimal r that minimises number of detectors but gives good discrimination
- if r is small like r=2: it is more general it covers more of the problem space
    - discrimination is bad
- if r = l the matching is completely specific
    - if r is specific you need to generate more detectors to cover the problem space
started at 24 because its double 12 which is what the original LYSIS paper used
the amount of coverage (the number of possible strings that a single detector matches) diminishes exponentially as r increases
T(l) for r = 24 is 1.208923729944402074796032 × 10^24
D(l) for r = 24 is 2.089670227099910144 × 10^18
D(l) for r = 17 is 2.67477789068788498432 × 10^20
D(l) for r = 12 is 8.559289250201231949824 × 10^21 matched most detectors had to increase asking myself if randomised detectors were a good idea or if regex to set limits
-> question wont the detector just look like impossible values for a packet? is that why we need a lot of them? no because r creates a pattern by avoiding normal patterns so the randomness descreases -> it starts avoiding the patterns int the target set """
    #TODO put a __repr__ method here
    def __new__(cls, *args, **kwargs):
       # print("Created a new instance of antibody.")
        return super().__new__(cls)


    def __init__(self):
        #TODO : how long is a packet with my fields and how to organise bits consistently
        #TODO : either find average number of ones in first 3 fields of IP packets
        # or find a function that produces random values in the range of what is normal for the 3 fields
        # TODO : generate different random bit arrays for different fields so the number of ones corresponds to the number of ones likely in the feild
        """or by predetermining the number of ones I am restricting learning"""
        detector = bitarray.bitarray()
        detector.frombytes(random.getrandbits(80).to_bytes(10, byteorder = "big"))
        self.detector = np.array(detector.tolist(), dtype=np.bool_)
        """number of detector activations with the most recent packet collection"""
        self.number_of_activations : int = 0
        self.r : int = 40
        self.was_activated_by_packetnr : list[int] = []


    def __repr__(self) -> str:
        return (
            f"{type(self).__name__}("
            f"detector={self.detector}, "
            f"number_of_activations={self.number_of_activations}, "
            f")"
        )


    """performs r contiguous bits comparison on each packet """


    #TODO : test if it returns true when r contiguous bits match
    #TODO : test if it returns false when r contiguous bits dont match
    def is_detector_activated(self, packet: NDArray[np.bool_]) -> bool:
        consecutive_matches = 0
        for bit, detector_bit in zip(packet,self.detector):
            if np.equal(bit, detector_bit) and consecutive_matches <= self.r:
                consecutive_matches += 1
            if consecutive_matches >= self.r:
                return True
            if not np.equal(bit, detector_bit):
                consecutive_matches = 0
        else:
            return False


    """this function takes packet collections and returns the number of activations it also modifies the attributes of antibody number_of_detector_activations and should_be_deleted_negative_selection"""
    #TODO : provide the information which packet it activated with
    def number_of_detector_activations(self, packet_collection: NDArray[np.bool_]) ->  int:
        number_of_detector_activations = 0
        packet_number = 0
        if packet_collection.ndim != 2:
            raise ValueError("packet_collection must be a 2D NumPy array")
        for packet in packet_collection:
            packet_number += 1
            if self.is_detector_activated(packet):
               number_of_detector_activations += 1
               self.was_activated_by_packetnr.append(packet_number)
        self.number_of_activations = number_of_detector_activations
        """"we say detectors work when number of activation similar to number of malicious packets in the collection"""
        return number_of_detector_activations

    def should_be_negatively_selected(self):
        if self.number_of_activations >0:
            return True
        else:
            return False

class Lymphocyte:

    def __new__(cls, *args, **kwargs):
        #print("Created a new instance of Lymphocyte.")
        #super allows access to parent class object we give its constructor
        #lymphocyte as an argument
        #we use a double underscore because line 7 calls a special method
        return super().__new__(cls)

    #self holds a reference to the current instance
    #TODO : plan is to create multiple antibodies per lymphocyte
    def __init__(self,self_set : NDArray[np.bool_]):
        #TODO: change single antibody an antibody array
        self.antibody = Antibody()
        self.antibody.number_of_detector_activations(self_set)
        #TODO : track how many times while loop is repeated for measure of specificity
        while self.antibody.should_be_negatively_selected():
               # print(f"rejected antibody", self.antibody.detector)
                self.antibody = Antibody()
                self.antibody.number_of_detector_activations(self_set)
        #print(f"accepted antibody", self.antibody.detector)
        #TODO : include threshold in activation
        self.threshold = 0

    def __repr__(self) -> str:
        return f"{type(self).__name__}(antibody = {self.antibody.detector})"

    """activation threshold of lymphocyte is exceeded when lymphocyte detects X number of antigens in a short period of time"""

    def is_threshold_exceeded(self):
        pass

    """"If lymphocyte is not unique it is discarded"""

    def is_lymphocyte_unique(self):
        pass


lymphocyte = Lymphocyte(Self_Set_Packets)
repr(lymphocyte)
repr(lymphocyte.antibody)

'Antibody(detector=[ True False  True False False  True  True False False False  True False\n False False  True  True False False False False False False  True False\n False False  True  True False  True False False False  True  True  True\n False  True  True  True False  True False  True False False False False\n False False False  True False False False  True  True False  True False\n  True  True  True False  True  True False False  True  True False  True\n  True False  True  True  True False False False], number_of_activations=0, )'